# Lab 03: Machine Learning Pipelines & Evaluation Without Data Leakage
**Course:** Data Exploration and Visualization Laboratory (B.Tech Computer Engineering)

### Objectives:
1. Partition datasets using stratified train/test splitting
2. Build scikit-learn `Pipeline` and `ColumnTransformer` preprocessors
3. Train Random Forest classifiers and extract feature importances
4. Evaluate generalization performance (Accuracy, Precision, Recall, F1, Confusion Matrix)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load customer churn dataset
df = pd.read_csv('../data/customer_churn.csv')
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']
print("Features:", list(X.columns))
print("Class distribution:\n", y.value_counts())

In [ ]:
# Train/Test split BEFORE any fitting to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
print("Model pipeline fitted successfully on training split.")

In [ ]:
# Evaluate strictly on held-out test split
y_pred = pipeline.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))